# braid example: ERA5 monthly statistics

Compute global 2m air temperature statistics for every month of a year,
in parallel across Lambda workers.

**Dataset**: ERA5 hourly reanalysis, hosted publicly on S3 at `s3://era5-pds/`  
**Pattern**: embarrassingly parallel — each Lambda handles one month  
**Dependencies**: `xarray`, `zarr`, `s3fs`

> **Result size constraint**: braid returns results via SQS (small) or S3 (larger objects),
> but worker functions should return **small objects** — scalars, dicts, lists of numbers.
> Returning large arrays is slow and may hit memory/timeout limits.  
> If you need output arrays, write them to S3/zarr inside the worker and return only a status
> dict. See `era5_climatology_region_write.ipynb` for that pattern.

## Setup

```bash
uv add braid xarray zarr s3fs
```

AWS credentials must be configured (`aws configure` or environment variables).

In [ ]:
# /// script
# requires-python = ">=3.13"
# dependencies = [
#   "braid",
#   "xarray",
#   "zarr",
#   "s3fs",
#   "pandas",
#   "matplotlib",
# ]
# ///

In [ ]:
import logging
import os
logging.basicConfig(level=logging.INFO, format="%(message)s")

os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

import braid

# # Skip if .braid/config.toml already exists
# config = braid.init(
#     bucket="my-bucket",       # replace with your S3 bucket
#     region="us-west-2",
#     create_role=True,          # creates braid-worker-role with minimum permissions
#     memory_mb=1024,            # xarray + zarr needs more than the 512MB default
#     timeout_s=300,
# )
# print(f"Initialized: {config.worker_function_name} in {config.region}")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")

import braid

# # Skip if .braid/config.toml already exists
# config = braid.init(
#     bucket="carbonplan-share",       # replace with your S3 bucket
#     region="us-west-2",
#     create_role=True,          # creates braid-worker-role with minimum permissions
#     memory_mb=1024,            # xarray + zarr needs more than the 512MB default
#     timeout_s=300,
# )
# print(f"Initialized: {config.worker_function_name} in {config.region}")

In [ ]:
# arn = braid.sync()
# print(f"Layer: {arn}")

## 3. Define the worker function

This function runs on Lambda. It opens one month of ERA5 data directly from S3
and returns a **small dict of scalars** — not an array.

This is the right pattern for braid: workers read large data from S3, compute
something, and return only the summary. The data never crosses the Lambda
return path.

ERA5 layout on S3:
```
s3://era5-pds/{year}/{month:02d}/data/{variable}.zarr/
```

In [ ]:
def era5_monthly_stats(year_month: tuple[int, int]) -> dict:
    """Compute 2m temperature statistics for one month of ERA5 data."""
    import numpy as np
    import xarray as xr

    year, month = year_month
    url = f"s3://era5-pds/{year}/{month:02d}/data/air_temperature_at_2_metres.zarr"

    ds = xr.open_zarr(url, storage_options={"anon": True})
    t2m = ds["air_temperature_at_2_metres"]  # (time, lat, lon), Kelvin

    # Monthly mean field, then global area-weighted mean
    monthly_mean = t2m.mean("time0")
    weights = np.cos(np.deg2rad(monthly_mean.latitude))
    global_mean = float(monthly_mean.weighted(weights).mean())

    return {
        "year": year,
        "month": month,
        "global_mean_k": global_mean,
        "global_mean_c": global_mean - 273.15,
        "min_k": float(monthly_mean.min()),
        "max_k": float(monthly_mean.max()),
    }

## 4. Check payload size before dispatching

`dry_run=True` serializes the function and prints sizes without invoking Lambda.

In [ ]:
from braid import fan

year = 2020
inputs = [(year, m) for m in range(1, 13)]

fan(era5_monthly_stats, inputs, dry_run=True)

In [ ]:
inputs

## 5. Run in parallel

12 months → 12 Lambda invocations. Each runs independently and returns a dict.
Results come back in input order regardless of completion order.

**batch_size guidance**: each invocation takes ~30–60s (one month of ERA5),
so `batch_size=1` gives maximum parallelism with acceptable cold-start overhead.

In [ ]:
import time

t0 = time.time()
results = fan(
    era5_monthly_stats,
    inputs,
    batch_size=1,
    timeout_s=300,
)
elapsed = time.time() - t0

print(f"{len(results)} months processed in {elapsed:.1f}s")
print(f"(vs ~{len(inputs) * 45:.0f}s serial estimate)")

## 6. Inspect results

In [ ]:
import pandas as pd

df = pd.DataFrame(results).sort_values("month").reset_index(drop=True)
df["date"] = pd.to_datetime(df[["year", "month"]].assign(day=1))
df[["date", "global_mean_c", "min_k", "max_k"]]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["date"], df["global_mean_c"], marker="o", linewidth=2)
ax.fill_between(
    df["date"],
    df["min_k"] - 273.15,
    df["max_k"] - 273.15,
    alpha=0.15,
    label="min–max range",
)
ax.set_ylabel("Temperature (°C)")
ax.set_title(f"ERA5 global 2m temperature — {year}")
ax.legend()
plt.tight_layout()

## 7. Scale up: multiple years

Same pattern, more inputs. `max_concurrency` keeps Lambda invocations under
the default 1000 burst limit.

In [ ]:
multi_year_inputs = [
    (year, month)
    for year in range(2015, 2024)
    for month in range(1, 13)
]  # 108 months

results_multi = fan(
    era5_monthly_stats,
    multi_year_inputs,
    batch_size=1,
    memory_mb=1024,
    timeout_s=300,
    max_concurrency=100,
)

df_multi = pd.DataFrame(results_multi)
df_multi["date"] = pd.to_datetime(df_multi[["year", "month"]].assign(day=1))
df_multi = df_multi.sort_values("date").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_multi["date"], df_multi["global_mean_c"], linewidth=1.5)
ax.set_ylabel("Temperature (°C)")
ax.set_title("ERA5 global 2m temperature — 2015–2023")
plt.tight_layout()

## Workflow summary

```
first time           →  braid.init(...)  +  braid.sync()
add dependency       →  braid.sync()
change worker code   →  nothing (cloudpickle re-serializes on every call)
run again            →  braid.fan(...)
```

The Lambda worker loads your function from S3 on each invocation — no redeploy needed for code changes.